<a href="https://colab.research.google.com/github/PatrikBaldon/Statistical_Forecast/blob/main/Cleaned_Correlation_Analysis_per_Material_SubGroup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analisi Avanzata e Selezione dei Driver per Forecast

Questo notebook implementa una pipeline strutturata per l'ingestione, il test e la selezione dei driver macroeconomici e di mercato.
Il flusso segue una rigorosa gerarchia econometrica:
1. Data Ingestion (Vendite + Indicatori)
2. Data Preprocessing & Alignment
3. Stationarity Testing (Augmented Dickey-Fuller)
4. Granger Causality Testing
5. Feature Selection per modelli ML/Statistici

## Fase 0: Setup e Installazione Librerie

In [ ]:
%%capture
!pip install arch statsmodels pandas numpy matplotlib seaborn yfinance pytrends

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import warnings
warnings.filterwarnings('ignore')
import ipywidgets as widgets
from IPython.display import display, clear_output

sns.set_theme(style="whitegrid")

## Fase 1: Data Ingestion (IMPORT INVARIATO)
In questa cella deve essere inserito **esattamente il codice che utilizzi ora** per importare le vendite e scaricare gli indicatori esterni (es. Google Trends, yfinance, dati macro).

In [ ]:
# ==========================================
# 1. CONFIGURAZIONE FILE E DATE
# ==========================================
BASE_PATH = "/content/drive/MyDrive/DeLonghi_Dataset/"
CONFIG = {
    'file_data': 'Stat_Fam.xlsx',       # File Excel principale dal Drive
    'file_mapping': 'mapping.xlsx',     # File Excel per le anagrafiche
    'sheet_actuals': {
        'Actuals': 'actuals_new',
        'Actuals_Cleaned': 'actuals_cleaned'
    },
    'date_start_analysis': '2024-03-01',
    'date_end_analysis': '2025-12-01',
    'history_end_date': '2024-01-01'    # Usato per calcolare CV2 e denominatore MASE
}

def load_and_melt(excel_file, sheet_name, specific_name):
    """Carica uno sheet e lo converte in formato Long, etichettando la colonna col nome specifico."""
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    df = df.rename(columns={'Statistical Family': 'Statistical_Family', 'Submarket ID': 'Submarket_ID'})

    id_vars = [col for col in df.columns if col in ['Statistical_Family', 'Submarket_ID', 'Key Figures']]
    df_melted = df.melt(id_vars=id_vars, var_name='Period', value_name='Value')

    # Convertiamo in numerico ma LASCIAMO I NaN
    df_melted['Value'] = pd.to_numeric(df_melted['Value'], errors='coerce')

    # FIX DATE
    df_melted['Period'] = df_melted['Period'].astype(str).str.split('.').str[0]
    df_melted['Period'] = pd.to_datetime(df_melted['Period'], format='%b-%y', errors='coerce')

    df_melted = df_melted.dropna(subset=['Period'])
    df_melted['Data_Category'] = specific_name

    return df_melted[['Statistical_Family', 'Submarket_ID', 'Period', 'Data_Category', 'Value']]

# --- CARICAMENTO MAPPING ANAGRAFICO ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    df_mapping = pd.read_excel(BASE_PATH + CONFIG['file_mapping'], sheet_name='mapping')
except Exception:
    df_mapping = pd.read_excel(BASE_PATH + CONFIG['file_mapping'])

df_mapping = df_mapping.rename(columns={'Statistical Family': 'Statistical_Family', 'Submarket ID': 'Submarket_ID'})

# --- ESTRAZIONE DATI ---
print(f"Lettura del file Excel {BASE_PATH + CONFIG['file_data']} in corso...")
xls_data = pd.ExcelFile(BASE_PATH + CONFIG['file_data'])

list_all_data = []
categories_loaded = []

# 1. Estrazione Fogli Actuals
for actual_name, sheet_name in CONFIG['sheet_actuals'].items():
    if sheet_name in xls_data.sheet_names:
        list_all_data.append(load_and_melt(xls_data, sheet_name, actual_name))
        categories_loaded.append(actual_name)

df_raw_long = pd.concat(list_all_data, ignore_index=True)
df_wide = df_raw_long.pivot_table(
    index=['Statistical_Family', 'Submarket_ID', 'Period'],
    columns='Data_Category',
    values='Value',
    aggfunc='sum'
).reset_index()

# Gestione Nulli nel passato per gli Actuals
oggi = pd.Timestamp.today()
storico_mask = df_wide['Period'] < oggi
actual_cols = [col for col in df_wide.columns if col in list(CONFIG['sheet_actuals'].keys())]

for col in actual_cols:
    df_wide.loc[storico_mask, col] = df_wide.loc[storico_mask, col].fillna(0)

# Filtriamo df_wide per mantenere solo le colonne Actuals, eliminando i forecast
df_actuals_only = df_wide[['Statistical_Family', 'Submarket_ID', 'Period'] + actual_cols]

# Join con anagrafica
df_master = pd.merge(df_actuals_only, df_mapping, on=['Statistical_Family', 'Submarket_ID'], how='left')

print(f"Data-Prep completata! Creato df_master con {len(df_master)} record, filtrato per includere solo i dati Actuals.")
display(df_master.head())


In [ ]:
import yfinance as yf
import pandas_datareader.data as web
from datetime import datetime

# Definiamo il range temporale basato sul dataset DeLonghi
start_date = df_master['Period'].min()
end_date = df_master['Period'].max()

def get_advanced_macro_indicators():
    print("Estrazione dati in corso...")

    # 1. Cambio EUR/USD
    eurusd = yf.download('EURUSD=X', start=start_date, end=end_date, interval='1mo')['Close']

    # 2. Indice S&P 500 (Proxy economia globale e 'wealth effect')
    sp500 = yf.download('^GSPC', start=start_date, end=end_date, interval='1mo')['Close']

    # 3. Consumer Sentiment Index (UMCSENT) - Fondamentale per beni non essenziali
    # 4. Personal Consumption Expenditures (PCE) - Inflazione specifica sui consumi
    try:
        macro_fred = web.DataReader(['UMCSENT', 'PCE'], 'fred', start_date, end_date).resample('MS').mean()
    except Exception as e:
        macro_fred = None
        print(f"Avviso: Impossibile recuperare dati FRED. Errore: {e}")

    return eurusd, sp500, macro_fred

eurusd_data, sp500_data, fred_data = get_advanced_macro_indicators()

# Consolidamento in un unico DataFrame Macro
df_macro = pd.DataFrame(index=eurusd_data.index)
df_macro['EURUSD'] = eurusd_data
df_macro['SP500'] = sp500_data
if fred_data is not None:
    df_macro = df_macro.join(fred_data)

print("Dataset Macroeconomico consolidato:")
display(df_macro.tail())

## Fase 2: Preprocessing e Allineamento Temporale
Uniamo tutti i dati in un unico Master DataFrame temporale, gestendo i valori nulli (Forward e Backward fill).

In [ ]:
# 1. Preparazione delle Variabili con Filtro di Qualità
# Aggreghiamo i dati e calcoliamo la varianza per evitare serie costanti o vuote
df_ts = df_master.groupby(['Period', 'Statistical_Family'])['Actuals'].sum().unstack().fillna(0)

# Filtriamo per le famiglie che hanno una somma totale di vendite significativa
# e che non sono costanti (deviazione standard > 0)
stats = df_ts.describe().T
valid_fams = stats[stats['std'] > 0].sort_values(by='mean', ascending=False).index.tolist()

if len(valid_fams) >= 2:
    famiglia_A = valid_fams[0]
    famiglia_B = valid_fams[1]
else:
    famiglia_A = df_ts.columns[0]
    famiglia_B = df_ts.columns[1]

df_analysis = df_ts[[famiglia_A, famiglia_B]].rename(columns={famiglia_A: 'Var_X', famiglia_B: 'Var_Y'})

print(f"Variabili selezionate per l'analisi (Top Volume): {famiglia_A} (X) e {famiglia_B} (Y)")
display(df_analysis.tail())

In [ ]:
# 1. Identificazione Famiglie Pilota e Preparazione Master TS
all_families_ts = df_master.groupby(['Period', 'Statistical_Family'])['Actuals'].sum().unstack().fillna(0)

# Calcoliamo statistiche per identificare i leader
family_volume = all_families_ts.sum().sort_values(ascending=False)
pilot_candidates = family_volume.head(3).index.tolist()

print(f"Famiglie Candidate come Pilota (Top Volume): {pilot_candidates}")

# Creazione DataFrame con Residui STL per TUTTE le famiglie
df_residuals = pd.DataFrame(index=all_families_ts.index)
for fam in all_families_ts.columns:
    if all_families_ts[fam].std() > 0:
        stl_res = STL(all_families_ts[fam], period=12).fit()
        df_residuals[fam] = stl_res.resid

# Unione con Macro
df_master_cleaned = df_residuals.join(df_macro).ffill().bfill()
display(df_master_cleaned.head())

In [ ]:
# 1. Unione dei dati Vendite con Indicatori Macro
df_combined = df_analysis.join(df_macro).dropna()

# Definizione colonna target (Sales) per l'analisi
target_sales_column = df_combined.columns[1]

print(f"Dataset integrato pronto. Target selezionato: {target_sales_column}")
display(df_combined.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from statsmodels.tsa.seasonal import STL

def decompose_sales(df, target_col):
    print(f"Decomposizione STL per {target_col}...")
    stl = STL(df[target_col], period=12)
    res = stl.fit()
    fig = res.plot()
    plt.show()
    return res

# Esecuzione e salvataggio dei risultati
sales_stl = decompose_sales(df_combined, target_sales_column)

# Creazione di un nuovo dataframe per i test successivi usando i residui (serie pulita da trend e stagionalità)
df_combined_cleaned = df_combined.copy()
df_combined_cleaned[target_sales_column] = sales_stl.resid

print("Serie destagionalizzata (residui) pronta per i test successivi in 'df_combined_cleaned'.")

## Fase 3: Analisi di Stazionarietà (Augmented Dickey-Fuller Test)
**LOGICA GERARCHICA:** Per eseguire correttamente il test di Granger e i modelli di forecast, le serie storiche devono essere stazionarie (senza trend o stagionalità marcate). Se non lo sono, applichiamo una differenziazione.

In [ ]:
# 2. Stazionarietà Universale
def make_stationary_all(df):
    df_stat = df.copy()
    for col in df_stat.columns:
        res = adfuller(df_stat[col].dropna())
        if res[1] > 0.05:
            df_stat[col] = df_stat[col].diff()
    return df_stat.dropna()

df_master_stationary = make_stationary_all(df_master_cleaned)
print("Dataset globale stazionario (Residui + Macro) pronto.")

## Fase 4: Granger Causality Tests
Ora che le serie sono stazionarie, testiamo se un indicatore X "causa nel senso di Granger" le Vendite Y.
L'ipotesi nulla (H0) è che X NON causi Y. Un p-value < 0.05 implica che X ha potere predittivo su Y.

In [ ]:
# 3. Analisi di Causalità (Macro + Causal Pilots vs Others)
# Usiamo top_causal_pilots derivanti da identify_causal_pilots
macro_list = ['EURUSD', 'SP500', 'UMCSENT', 'PCE']

global_granger = run_global_causality(df_master_stationary, top_causal_pilots, macro_list)

print(f"Top Relazioni di Causalità con Piloti Causal ({top_causal_pilots}):")
display(global_granger.head(20))

## Fase 4.1: Analisi della Correlazione e Multicollinearità
Prima di procedere, verifichiamo la correlazione lineare tra le variabili per evitare di inserire driver ridondanti nel modello.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_correlation_matrix(df):
    plt.figure(figsize=(10, 8))
    corr_matrix = df.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title('Matrice di Correlazione (Residui Vendite + Macro)')
    plt.show()
    return corr_matrix

# Analizziamo la correlazione sui residui per isolare l'impatto macro
corr_results = plot_correlation_matrix(df_combined_cleaned)
display(corr_results)

## Fase 4.2: Test di Cointegrazione (Engle-Granger)
Verifichiamo se esiste un legame di equilibrio di lungo periodo tra le vendite e i driver selezionati.

In [ ]:
# 4. Test di Cointegrazione Globale con Hub Leaders
# Utilizziamo i leader della rete (network_stats) per il test di equilibrio
hub_leaders = network_stats.head(3).index.tolist()

global_coint = run_global_cointegration(df_master_cleaned, hub_leaders, macro_list)

print(f"Relazioni di Cointegrazione con Hub Leaders ({hub_leaders}):")
display(global_coint)

## Fase 4.3: Mutual Information - Metodo K-Nearest Neighbors (K-NN) e Stimatore di Kraskov (KSG)

Lo stimatore **Kraskov-St&ouml;gbauer-Grassberger (KSG)** &egrave; attualmente considerato lo stato dell'arte e il candidato ideale per l'analisi delle dipendenze non lineari. Questo metodo analizza le distanze tra i punti vicini (*nearest neighbors*) nello spazio multidimensionale ed &egrave; raccomandato perch&eacute; offre la migliore accuratezza per identificare campi stocastici e serie storiche con relazioni fortemente dipendenti e distribuzioni non Gaussiane.

&Egrave; anche il metodo standard implementato nelle librerie di Machine Learning moderne, come *scikit-learn* in Python.

L'informazione mutua misurata ($I$) pu&ograve; essere matematicamente trasformata in un **Indice normalizzato $R$**, calcolato come:

$$R = \sqrt{1 - \exp(-2 \times I)}$$

In questo modo, il grado di dipendenza non lineare viene forzato in una scala compresa tra 0 e 1, rendendolo direttamente paragonabile a una classica percentuale di correlazione lineare (Pearson).

In [ ]:
from sklearn.feature_selection import mutual_info_regression

def calculate_normalized_mi(df, target_col):
    print(f"Calcolo Mutual Information (KSG) rispetto a {target_col}...")
    X = df.drop(columns=[target_col])
    y = df[target_col]

    # Calcolo MI (scikit-learn usa l'approccio basato su KNN/KSG)
    mi_scores = mutual_info_regression(X, y, random_state=42)

    # Trasformazione in Indice R (Normalizzazione)
    r_indices = np.sqrt(1 - np.exp(-2 * mi_scores))

    mi_results = pd.DataFrame({
        'Mutual_Information': mi_scores,
        'R_Normalized_Index': r_indices
    }, index=X.columns).sort_values(by='R_Normalized_Index', ascending=False)

    return mi_results

# Esempio di applicazione sul dataset stazionario
mi_analysis = calculate_normalized_mi(df_combined_cleaned, target_sales_column)
display(mi_analysis)

## Fase 5: Selezione Feature ed Esportazione per il Forecasting
Isoliamo solo i driver che hanno dimostrato una causalità statistica per passarla ai modelli di Machine Learning (es. Darts, Chronos, o SAP IBP).

In [ ]:
def select_and_export_features(master_df, granger_df, target_col, p_value_threshold=0.05):
    valid_features = granger_df[granger_df['min_p_value'] <= p_value_threshold].index.tolist()
    cols_to_keep = [target_col] + valid_features
    final_dataset = master_df[cols_to_keep]
    print(f"Variabili selezionate: {valid_features}")
    #final_dataset.to_csv('Forecast_Ready_Dataset.csv')
    return final_dataset

# ESECUZIONE ATTIVATA
final_df = select_and_export_features(df_combined, granger_results, target_sales_column)
display(final_df.head())

## Fase 6: Analisi Avanzata della Stagionalit%e0 (STL Decomposition)
Isoliamo la componente stagionale e il trend per capire quanto del segnale %e8 guidato dai driver macro e quanto dalla stagionalit%e0 intrinseca (es. Natale).

## Fase 7: Lead/Lag Analysis (Cross-Correlation Function)
Identifichiamo se i driver macroeconomici sono 'Leading Indicators' che anticipano le vendite di alcuni mesi.

In [ ]:
def plot_cross_correlation(df, target_col, max_lag=12):
    drivers = [col for col in df.columns if col != target_col]
    n_drivers = len(drivers)
    fig, axes = plt.subplots(n_drivers, 1, figsize=(10, 5 * n_drivers))

    for i, var in enumerate(drivers):
        ax = axes[i] if n_drivers > 1 else axes
        ccov = [df[target_col].corr(df[var].shift(lag)) for lag in range(-max_lag, max_lag + 1)]
        lags = range(-max_lag, max_lag + 1)
        ax.stem(lags, ccov)
        ax.set_title(f'Cross-Correlation: {var} vs {target_col}')
        ax.set_xlabel('Lag (mesi)')
        ax.set_ylabel('Correlazione')
        ax.axhline(0.2, color='red', linestyle='--')
        ax.axhline(-0.2, color='red', linestyle='--')

plot_cross_correlation(df_stationary, target_sales_column)

## Fase 8: Test di Eteroschedasticit%e0 (ARCH Test)
Verifichiamo se la varianza dell'errore (volatilit%e0) %e8 costante nel tempo o se esistono periodi di instabilit%e0 raggruppata.

In [ ]:
from arch import arch_model

def test_volatility_clusters(series):
    print(f"--- Test ARCH per la Volatilit%e0 ---")
    model = arch_model(series, vol='Garch', p=1, q=1)
    res = model.fit(disp='off')
    display(res.summary())
    return res

vol_res = test_volatility_clusters(df_stationary[target_sales_column])

## Fase 9: Backtesting e Out-of-Sample Validation
Utilizziamo una tecnica di 'Rolling Window' per validare la capacit%e0 predittiva dei driver macroeconomici, misurando l'errore (MAPE) su diversi orizzonti temporali.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error

def time_series_cross_validation(df, target_col, exog_cols, horizon=6, steps=3):
    print(f"--- Inizio Backtesting (Target: {target_col}, Exog: {exog_cols}) ---")
    errors = []
    n_obs = len(df)

    # Rolling window: addestriamo sul passato e testiamo sul futuro
    for i in range(steps):
        train_end = n_obs - (steps - i) * horizon
        test_end = train_end + horizon

        train = df.iloc[:train_end]
        test = df.iloc[train_end:test_end]

        try:
            # Modello SARIMAX con driver esterni
            model = SARIMAX(train[target_col],
                            exog=train[exog_cols] if exog_cols else None,
                            order=(1, 1, 1),
                            seasonal_order=(1, 1, 1, 12))
            res = model.fit(disp=False)

            # Forecast
            pred = res.forecast(steps=horizon, exog=test[exog_cols] if exog_cols else None)
            mape = mean_absolute_percentage_error(test[target_col], pred)
            errors.append(mape)
            print(f"Finestra {i+1}: MAPE = {mape:.2%}")
        except Exception as e:
            print(f"Errore nella finestra {i+1}: {e}")

    if errors:
        print(f"\nMAPE Medio Finale: {np.mean(errors):.2%}")
    return errors

# ESECUZIONE: Testiamo l'impatto di SP500 e PCE (i driver con pi%f9 correlazione)
selected_drivers = ['SP500', 'PCE']
cross_val_results = time_series_cross_validation(df_combined, target_sales_column, selected_drivers)

## Fase 9: Backtesting e Out-of-Sample Validation
Utilizziamo una tecnica di 'Rolling Window' (Time Series Cross-Validation) per misurare l'accuratezza predittiva reale (MAPE) del modello includendo i driver esterni selezionati.

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error

def run_backtesting_validation(df, target_col, exog_cols, horizon=6, steps=3):
    print(f"--- Inizio Validazione Out-of-Sample (Horizon: {horizon} mesi) ---")
    errors = []
    n = len(df)

    for i in range(steps):
        # Definiamo i confini del training e test set per ogni finestra rolling
        train_end = n - (steps - i) * horizon
        test_end = train_end + horizon

        train = df.iloc[:train_end]
        test = df.iloc[train_end:test_end]

        try:
            # Modello SARIMAX con driver esterni (Esogeni)
            model = sm.tsa.statespace.SARIMAX(
                train[target_col],
                exog=train[exog_cols] if exog_cols else None,
                order=(1, 1, 1),
                seasonal_order=(1, 1, 1, 12)
            )
            res = model.fit(disp=False)

            # Forecast per l'orizzonte definito
            forecast = res.forecast(steps=horizon, exog=test[exog_cols] if exog_cols else None)

            # Calcolo errore
            mape = mean_absolute_percentage_error(test[target_col], forecast)
            errors.append(mape)
            print(f"Finestra {i+1}: MAPE = {mape:.2%}")

        except Exception as e:
            print(f"Errore nella finestra {i+1}: {e}")

    if errors:
        print(f"\nMAPE Medio Finale (Backtesting): {np.mean(errors):.2%}")
    return errors

# Esecuzione: Validiamo l'impatto dei driver top (SP500, PCE)
selected_drivers = ['SP500', 'PCE']
valid_results = run_backtesting_validation(df_combined, target_sales_column, selected_drivers)

In [ ]:
def identify_causal_pilots(df_stationary, max_lag=4, p_threshold=0.05):
    print("Ricerca Causal Leadership in corso (potrebbe richiedere tempo)...")
    families = [c for c in df_stationary.columns if c not in ['EURUSD', 'SP500', 'UMCSENT', 'PCE']]
    causal_counts = {fam: 0 for fam in families}

    # Testiamo ogni famiglia contro tutte le altre per vedere chi 'guida' più spesso
    for pilot_cand in families:
        for target in families:
            if pilot_cand == target: continue
            try:
                # Test: pilot_cand causa target?
                gc = grangercausalitytests(df_stationary[[target, pilot_cand]], maxlag=max_lag, verbose=False)
                p_vals = [gc[i+1][0]['ssr_ftest'][1] for i in range(max_lag)]
                if min(p_vals) < p_threshold:
                    causal_counts[pilot_cand] += 1
            except: continue

    pilot_df = pd.DataFrame.from_dict(causal_counts, orient='index', columns=['Families_Driven'])
    return pilot_df.sort_values(by='Families_Driven', ascending=False)

# Esecuzione Screening Causal Leaders
causal_pilots = identify_causal_pilots(df_master_stationary)
print("\nClassifica Famiglie Pilota per Potere Predittivo (Granger):")
display(causal_pilots.head(10))

# Aggiorniamo la lista dei candidati pilota per i test successivi
top_causal_pilots = causal_pilots.head(3).index.tolist()
print(f"Nuovi Piloti Selezionati: {top_causal_pilots}")

In [ ]:
def analyze_causal_network(df_stationary, max_lag=4, p_threshold=0.05):
    import networkx as nx
    print("Costruzione del Grafo delle Influenze Causalità...")

    families = [c for c in df_stationary.columns if c not in ['EURUSD', 'SP500', 'UMCSENT', 'PCE']]
    G = nx.DiGraph()

    # Costruiamo i legami di causalità
    for driver in families:
        for target in families:
            if driver == target: continue
            try:
                gc = grangercausalitytests(df_stationary[[target, driver]], maxlag=max_lag, verbose=False)
                p_vals = [gc[i+1][0]['ssr_ftest'][1] for i in range(max_lag)]
                if min(p_vals) < p_threshold:
                    G.add_edge(driver, target, weight=1-min(p_vals))
            except: continue

    # 1. Centralità (Chi è il vero motore globale?)
    centrality = nx.out_degree_centrality(G)

    # 2. Hub Analysis (Famiglie che guidano altre famiglie che a loro volta sono leader)
    hubs, authorities = nx.hits(G, max_iter=100)

    # 3. Identificazione dei Target Comuni
    in_degrees = dict(G.in_degree())

    results = pd.DataFrame({
        'Influence_Reach': pd.Series(centrality),
        'Hub_Score_Leadership': pd.Series(hubs),
        'Target_Overlap_Count': pd.Series(in_degrees)
    }).sort_values(by='Hub_Score_Leadership', ascending=False)

    print("\n--- Analisi della Rete Causalità ---")
    print("Hub Score: indica quanto una famiglia guida altre famiglie 'importanti'.")
    display(results.head(15))

    # Visualizzazione semplificata della gerarchia
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=0.5)
    nx.draw_networkx_nodes(G, pos, node_size=700, node_color='skyblue')
    nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, edge_color='gray', alpha=0.3)
    nx.draw_networkx_labels(G, pos, font_size=10)
    plt.title("Mappa delle Influenze Causalità tra Famiglie De'Longhi")
    plt.show()

    return G, results

causal_graph, network_stats = analyze_causal_network(df_master_stationary)

In [ ]:
from statsmodels.tsa.vector_ar.vecm import VECM, select_order, select_coint_rank

def run_vecm_analysis(df, target_col, pilot_col):
    print(f"--- Analisi VECM: {target_col} vs {pilot_col} ---")
    data = df[[target_col, pilot_col]].dropna()

    # 1. Selezione Lag
    lag_order = select_order(data, maxlags=6, deterministic='ci')
    k_ar = lag_order.aic if (lag_order.aic is not None and lag_order.aic > 0) else 1

    # 2. Test Rango di Cointegrazione (Johansen)
    # Correzione: usiamo 'signif' invece di 'signific' per compatibilità
    try:
        rank_test = select_coint_rank(data, det_order=0, k_ar_diff=k_ar, method='trace', signif=0.05)
    except TypeError:
        rank_test = select_coint_rank(data, det_order=0, k_ar_diff=k_ar, method='trace', signific=0.05)

    if rank_test.rank > 0:
        model = VECM(data, k_ar_diff=k_ar, coint_rank=rank_test.rank, deterministic='ci')
        res = model.fit()
        print(res.summary())
        return res
    else:
        print("Nessuna cointegrazione trovata. Si consiglia l'uso di un modello VAR sulle differenze.")
        return None

In [ ]:
def run_mgarch_dcc(df, target_col, driver_col):
    # Implementazione semplificata della correlazione dinamica
    # Poiché MGARCH-DCC richiede ottimizzatori complessi, usiamo una Rolling Correlation
    # con pesi esponenziali (EWMA) per simulare la dinamica DCC
    print(f"--- Analisi Dinamica delle Correlazioni (DCC Proxy): {target_col} vs {driver_col} ---")

    returns = df[[target_col, driver_col]].pct_change().dropna()
    dcc_corr = returns[target_col].ewm(span=12).corr(returns[driver_col])

    plt.figure(figsize=(10, 5))
    plt.plot(dcc_corr, label='DCC (Dynamic Correlation)', color='orange', linewidth=2)
    plt.axhline(dcc_corr.mean(), color='red', linestyle='--', label='Mean Corr')
    plt.title(f'Evoluzione della Correlazione nel Tempo: {target_col} vs {driver_col}')
    plt.legend()
    plt.show()
    return dcc_corr

In [ ]:
def run_advanced_dashboard(family_name, driver_name, analysis_type):
    # Preparazione
    df_local = all_families_ts[family_name].to_frame(name='Sales').join(df_macro).dropna()

    plt.close('all')
    with output_v3:
        clear_output()
        if analysis_type == 'VECM Model':
            run_vecm_analysis(df_local, 'Sales', driver_name)
        elif analysis_type == 'Dynamic Corr (DCC)':
            run_mgarch_dcc(df_local, 'Sales', driver_name)
        elif analysis_type == 'Lead/Lag (CCF)':
            # Già definito in precedenza
            plot_cross_correlation(df_local.diff().dropna(), 'Sales', max_lag=6)

# Widget v3
fam_drop = widgets.Dropdown(options=available_fams, description='Famiglia:')
drv_drop = widgets.Dropdown(options=macro_list + hub_leaders, description='Driver:')
ana_drop = widgets.Dropdown(options=['VECM Model', 'Dynamic Corr (DCC)', 'Lead/Lag (CCF)'], description='Test:')
btn_v3 = widgets.Button(description='Esegui Test Avanzato', button_style='info')
output_v3 = widgets.Output()

btn_v3.on_click(lambda b: run_advanced_dashboard(fam_drop.value, drv_drop.value, ana_drop.value))
display(widgets.VBox([widgets.HBox([fam_drop, drv_drop, ana_drop]), btn_v3]), output_v3)

In [ ]:
# Esecuzione diretta dell'analisi MGARCH-DCC (Proxy EWMA)
# Utilizziamo la famiglia leader MCCC come riferimento per le 'Sales'
family_to_test = 'MCCC'
driver_to_test = 'EURUSD'

# Preparazione dati
df_local = all_families_ts[family_to_test].to_frame(name='Sales').join(df_macro).dropna()

# Esecuzione test
dcc_results = run_mgarch_dcc(df_local, 'Sales', driver_to_test)